In [ ]:
%pip install --upgrade azure-ai-ml azure-identity

In [ ]:
from pathlib import Path
import py_compile

candidates = [
    Path.cwd(),
    Path.cwd() / "ralir",
    Path("/home/azureuser/cloudfiles/code/Users/estherxin0011/ralir"),
]

PROJECT_DIR = next(
    (
        path.resolve()
        for path in candidates
        if (path / "job_code" / "ralir_job.py").is_file()
    ),
    None,
)

assert PROJECT_DIR is not None, (
    f"Could not locate the project. Current directory: {Path.cwd()}"
)

WORKER = PROJECT_DIR / "job_code" / "ralir_job.py"

assert WORKER.stat().st_size > 0, "ralir_job.py is empty"
py_compile.compile(str(WORKER), doraise=True)

print("PASS: project directory =", PROJECT_DIR)
print("PASS: worker =", WORKER)
print("PASS: worker syntax is valid")

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION = "7a8513b6-ada2-4ad1-aee2-687fa5663c82"
RESOURCE_GROUP = "AIModels"
WORKSPACE = "Reinforcementinfra"
ASSET_VERSION = "1"

client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WORKSPACE,
)

workspace = client.workspaces.get(WORKSPACE)
print("PASS: workspace =", workspace.name)

for name in ["ralir-dpr-raw", "ralir-provenance"]:
    asset = client.data.get(name=name, version=ASSET_VERSION)
    print("PASS: data =", asset.name, asset.version, asset.type)

for name in [
    "ralir-colbertv2",
    "ralir-reward-qwen",
    "ralir-eval-phi",
    "ralir-e5",
]:
    asset = client.models.get(name=name, version=ASSET_VERSION)
    print("PASS: model =", asset.name, asset.version, asset.type)

environment = client.environments.get(
    name="colbertv2-gpu-env",
    version="1",
)
compute = client.compute.get("t4v3")
datastore = client.datastores.get("workspaceblobstore")

print("PASS: environment =", environment.name, environment.version)
print("PASS: compute =", compute.name, compute.provisioning_state)
print("PASS: datastore =", datastore.name)

In [ ]:
from azure.ai.ml import command
from pathlib import Path
import textwrap
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml.entities import CommandJobLimits
from azure.identity import DefaultAzureCredential

SUBSCRIPTION = "7a8513b6-ada2-4ad1-aee2-687fa5663c82"
RESOURCE_GROUP = "AIModels"
WORKSPACE = "Reinforcementinfra"
COMPUTE = "t4v3"
ENVIRONMENT = "azureml:colbertv2-gpu-env:5"
ASSET_VERSION = "1"

# Keep this stable while resuming the same pilot.
MODE = "pilot"
RUN_ID = "pilot-20260822-03"

# Run only part of the workflow when debugging.
START_STEP = 11
END_STEP = 11

ROOT = (
    "azureml://datastores/workspaceblobstore/"
    f"paths/ralir/runs/{RUN_ID}"
)
METRICS = f"{ROOT}/metrics"
RANKINGS = f"{ROOT}/rankings"

possible_project_dirs = [
    Path.cwd(),
    Path.cwd() / "ralir",
    Path("/home/azureuser/cloudfiles/code/Users/estherxin0011/ralir"),
]

PROJECT_DIR = next(
    (
        path.resolve()
        for path in possible_project_dirs
        if (path / "job_code" / "ralir_job.py").is_file()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Cannot find Users/estherxin0011/ralir/job_code/ralir_job.py. "
        f"Current directory is {Path.cwd()}"
    )

CODE_DIR = PROJECT_DIR / "job_code"

print("Project directory:", PROJECT_DIR)
print("Worker code:", CODE_DIR / "ralir_job.py")

smoke_job = command(
    display_name="ralir-env-v3-smoke",
    experiment_name="ralir-environment-validation",
    code=str(CODE_DIR),
    command=(
        'python -c "'
        "import sys, torch, colbert, faiss; "
        "from importlib.metadata import version; "
        "print('python=', sys.executable); "
        "print('torch=', torch.__version__); "
        "print('torch_cuda=', torch.version.cuda); "
        "print('gpu=', torch.cuda.get_device_name(0)); "
        "print('colbert_ai=', version('colbert-ai')); "
        "print('colbert_module=', colbert.__file__); "
        "print('faiss_package=', version('faiss-gpu-cu12')); "
        "print('faiss=', faiss.__version__); "
        "assert torch.cuda.is_available(); "
        "assert hasattr(faiss, 'StandardGpuResources'); "
        "faiss.StandardGpuResources(); "
        "print('SMOKE TEST PASSED')"
        '"'
    ),
    environment="azureml:colbertv2-gpu-env:3",
    compute="t4v3",
)

created = client.jobs.create_or_update(smoke_job)
print(created.studio_url)
client.jobs.stream(created.name)

In [3]:
from pathlib import Path
import py_compile
from pathlib import Path
import textwrap
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml.entities import CommandJobLimits
from azure.identity import DefaultAzureCredential

SUBSCRIPTION = "7a8513b6-ada2-4ad1-aee2-687fa5663c82"
RESOURCE_GROUP = "AIModels"
WORKSPACE = "Reinforcementinfra"
COMPUTE = "t4v3"
ENVIRONMENT = "azureml:colbertv2-gpu-env:5"
ASSET_VERSION = "1"

# Keep this stable while resuming the same pilot.
MODE = "pilot"
RUN_ID = "pilot-20260823-01"
worker = Path(
    "/home/azureuser/cloudfiles/code/Users/"
    "estherxin0011/ralir/job_code/ralir_job.py"
)

py_compile.compile(str(worker), doraise=True)

source = worker.read_text(encoding="utf-8")
start = source.index("def load_adapter(path):")
end = source.index("\ndef load_encoder", start)
adapter_code = source[start:end]

print(adapter_code)
assert "adapter.to(" in adapter_code

print("PASS: updated worker is valid")

def load_adapter(path):
    checkpoint_path = find_file(path, "adapter.pt")
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    adapter = LowRankQueryAdapter(
        checkpoint["dimension"],
        checkpoint["rank"],
    )
    adapter.load_state_dict(checkpoint["state_dict"])

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    adapter.to(device=device, dtype=torch.float32)
    adapter.eval()

    print(
        f"Loaded adapter on {next(adapter.parameters()).device}: "
        f"{checkpoint_path}",
        flush=True,
    )
    return adapter


PASS: updated worker is valid


In [1]:
from pathlib import Path
import textwrap
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml.entities import CommandJobLimits
from azure.identity import DefaultAzureCredential

SUBSCRIPTION = "7a8513b6-ada2-4ad1-aee2-687fa5663c82"
RESOURCE_GROUP = "AIModels"
WORKSPACE = "Reinforcementinfra"
COMPUTE = "t4v3"
ENVIRONMENT = "azureml:colbertv2-gpu-env:5"
ASSET_VERSION = "1"

# Keep this stable while resuming the same pilot.
MODE = "pilot"
RUN_ID = "pilot-20260823-01"

# Run only part of the workflow when debugging.
START_STEP = 20
END_STEP = 20
STEP20_ATTEMPT = "retry01"

ROOT = (
    "azureml://datastores/workspaceblobstore/"
    f"paths/ralir/runs/{RUN_ID}"
)
METRICS = f"{ROOT}/metrics"
RANKINGS = f"{ROOT}/rankings"

possible_project_dirs = [
    Path.cwd(),
    Path.cwd() / "ralir",
    Path("/home/azureuser/cloudfiles/code/Users/estherxin0011/ralir"),
]

PROJECT_DIR = next(
    (
        path.resolve()
        for path in possible_project_dirs
        if (path / "job_code" / "ralir_job.py").is_file()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Cannot find Users/estherxin0011/ralir/job_code/ralir_job.py. "
        f"Current directory is {Path.cwd()}"
    )

CODE_DIR = PROJECT_DIR / "job_code"

print("Project directory:", PROJECT_DIR)
print("Worker code:", CODE_DIR / "ralir_job.py")

client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WORKSPACE,
)


def data_asset(name):
    return Input(
        type="uri_folder",
        path=f"azureml:{name}:{ASSET_VERSION}",
        mode="download",
    )


def model_asset(name):
    return Input(
        type="custom_model",
        path=f"azureml:{name}:{ASSET_VERSION}",
        mode="download",
    )


def folder(path):
    return Input(
        type="uri_folder",
        path=path,
        mode="download",
    )


def output(path):
    return Output(
        type="uri_folder",
        path=path,
        mode="upload",
    )


def enabled(step):
    return START_STEP <= step <= END_STEP

def submit(display_name, job_command, inputs, outputs):
    normalized_command = " ".join(
        line.strip()
        for line in textwrap.dedent(job_command).splitlines()
        if line.strip()
    )

    print("Command:", normalized_command)
    assert "\n" not in normalized_command

    job = command(
        display_name=f"{RUN_ID}-{display_name}",
        experiment_name=f"ralir-{MODE}",
        code=str(CODE_DIR),
        command=normalized_command,
        inputs=inputs,
        outputs=outputs,
        environment=ENVIRONMENT,
        compute=COMPUTE,
        limits=CommandJobLimits(timeout=604800),
        environment_variables={
            "HF_HUB_OFFLINE": "1",
            "TRANSFORMERS_OFFLINE": "1",
            "TOKENIZERS_PARALLELISM": "false",
            "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        },
        tags={
            "project": "reward-aligned-colbert",
            "mode": MODE,
            "run_id": RUN_ID,
        },
    )

    created = client.jobs.create_or_update(job)
    print(f"Submitted {display_name}: {created.name}")
    print("Studio URL:", created.studio_url)

    client.jobs.stream(created.name)

    final_job = client.jobs.get(created.name)
    if final_job.status != "Completed":
        raise RuntimeError(
            f"{display_name} ended with status {final_job.status}"
        )

    return final_job


# Preflight
print("Workspace:", client.workspace_name)

for asset_name in ["ralir-dpr-raw", "ralir-provenance"]:
    asset = client.data.get(asset_name, version=ASSET_VERSION)
    print("DATA:", asset.name, asset.version, asset.type)

for asset_name in [
    "ralir-colbertv2",
    "ralir-reward-qwen",
    "ralir-eval-phi",
    "ralir-e5",
]:
    asset = client.models.get(asset_name, version=ASSET_VERSION)
    print("MODEL:", asset.name, asset.version, asset.type)

environment = client.environments.get(
    name="colbertv2-gpu-env",
    version="5",
)
compute = client.compute.get(COMPUTE)
datastore = client.datastores.get("workspaceblobstore")

print("ENVIRONMENT:", environment.name, environment.version)
print("COMPUTE:", compute.name, compute.provisioning_state)
print("OUTPUT DATASTORE:", datastore.name)

pilot = MODE == "pilot"

passages = 200_000 if pilot else 0
train_queries = 200 if pilot else 5_000
utility_queries = 200 if pilot else 5_000
retrieval_queries = 100 if pilot else 0
rag_queries = 50 if pilot else 1_000
candidate_k = 16
index_bits = [2] if pilot else [1, 2, 4]

raw = data_asset("ralir-dpr-raw")
provenance = data_asset("ralir-provenance")

colbert = model_asset("ralir-colbertv2")
qwen = model_asset("ralir-reward-qwen")
phi = model_asset("ralir-eval-phi")
e5 = model_asset("ralir-e5")

prepared_uri = f"{ROOT}/prepared"
candidate_uri = f"{ROOT}/candidates"
cache_uri = f"{ROOT}/candidate-cache"
labels_uri = f"{ROOT}/utility-labels-v2"

utility_adapter_uri = f"{ROOT}/adapters/utility"
supervised_adapter_uri = f"{ROOT}/adapters/supervised"

e5_index_uri = f"{ROOT}/indexes/e5"
rag_predictions_uri = f"{ROOT}/predictions/rag"

index_uris = {
    bits: f"{ROOT}/indexes/colbert-nbits{bits}"
    for bits in index_bits
}

colbert_ranking_uris = {
    bits: (
        f"{RANKINGS}/colbert-nbits{bits}-"
        f"{STEP20_ATTEMPT}"
    )
    for bits in index_bits
}

e5_rankings_uri = f"{RANKINGS}/e5"


# Step 11
if enabled(11):
    submit(
        "11-validate",
        """
        python ralir_job.py validate
        --raw ${{inputs.raw}}
        --colbert ${{inputs.colbert}}
        --qwen ${{inputs.qwen}}
        --phi ${{inputs.phi}}
        --e5 ${{inputs.e5}}
        --provenance ${{inputs.provenance}}
        --output ${{outputs.report}}
        """,
        inputs={
            "raw": raw,
            "colbert": colbert,
            "qwen": qwen,
            "phi": phi,
            "e5": e5,
            "provenance": provenance,
        },
        outputs={
            "report": output(f"{METRICS}/validation"),
        },
    )


# Step 12
if enabled(12):
    submit(
        "12-prepare-data",
        f"""
        python ralir_job.py prepare
        --raw ${{{{inputs.raw}}}}
        --output ${{{{outputs.prepared}}}}
        --mode {MODE}
        --pilot-passages {passages}
        --pilot-train-queries {train_queries}
        --pilot-eval-queries {retrieval_queries}
        --seed 13
        """,
        inputs={"raw": raw},
        outputs={"prepared": output(prepared_uri)},
    )


# Step 13
if enabled(13):
    for bits in index_bits:
        submit(
            f"13-index-{bits}bit",
            f"""
            python ralir_job.py index
            --prepared ${{{{inputs.prepared}}}}
            --model ${{{{inputs.model}}}}
            --output ${{{{outputs.index}}}}
            --metrics ${{{{outputs.metrics}}}}
            --nbits {bits}
            """,
            inputs={
                "prepared": folder(prepared_uri),
                "model": colbert,
            },
            outputs={
                "index": output(index_uris[bits]),
                "metrics": output(f"{METRICS}/index-{bits}bit"),
            },
        )


# Steps 14-15
if enabled(14) or enabled(15):
    submit(
        "14-15-candidates-cache",
        f"""
        python ralir_job.py candidates
        --prepared ${{{{inputs.prepared}}}}
        --index ${{{{inputs.index}}}}
        --model ${{{{inputs.model}}}}
        --candidates ${{{{outputs.candidates}}}}
        --cache ${{{{outputs.cache}}}}
        --max-queries {train_queries}
        --k {candidate_k}
        --seed 17
        """,
        inputs={
            "prepared": folder(prepared_uri),
            "index": folder(index_uris[2]),
            "model": colbert,
        },
        outputs={
            "candidates": output(candidate_uri),
            "cache": output(cache_uri),
        },
    )


# Step 16
if enabled(16):
    submit(
        "16-utility-labels",
        f"""
        python ralir_job.py utility
        --prepared ${{{{inputs.prepared}}}}
        --candidates ${{{{inputs.candidates}}}}
        --model ${{{{inputs.model}}}}
        --output ${{{{outputs.labels}}}}
        --max-queries {utility_queries}
        --batch-size 4
        --max-new-tokens 32
        """,
        inputs={
            "prepared": folder(prepared_uri),
            "candidates": folder(candidate_uri),
            "model": qwen,
        },
        outputs={
            "labels": output(labels_uri),
        },
    )


# Step 17
if enabled(17):
    submit(
        "17-train-utility",
        f"""
        python ralir_job.py train
        --cache ${{{{inputs.cache}}}}
        --labels ${{{{inputs.labels}}}}
        --output ${{{{outputs.adapter}}}}
        --metrics ${{{{outputs.metrics}}}}
        --objective utility
        --rank 16
        --epochs {2 if pilot else 3}
        --learning-rate 0.0003
        --lambda-nq 0.25
        --gamma-anchor 0.02
        --seed 17
        """,
        inputs={
            "cache": folder(cache_uri),
            "labels": folder(labels_uri),
        },
        outputs={
            "adapter": output(utility_adapter_uri),
            "metrics": output(f"{METRICS}/train-utility"),
        },
    )


# Step 18
if enabled(18):
    submit(
        "18-train-supervised",
        f"""
        python ralir_job.py train
        --cache ${{{{inputs.cache}}}}
        --labels ${{{{inputs.labels}}}}
        --output ${{{{outputs.adapter}}}}
        --metrics ${{{{outputs.metrics}}}}
        --objective supervised
        --rank 16
        --epochs {2 if pilot else 3}
        --learning-rate 0.0003
        --lambda-nq 1.0
        --gamma-anchor 0.02
        --seed 17
        """,
        inputs={
            "cache": folder(cache_uri),
            "labels": folder(labels_uri),
        },
        outputs={
            "adapter": output(supervised_adapter_uri),
            "metrics": output(f"{METRICS}/train-supervised"),
        },
    )


# Step 19
if enabled(19):
    submit(
        "19-e5-index",
        f"""
        python ralir_job.py e5-index
        --prepared ${{{{inputs.prepared}}}}
        --model ${{{{inputs.model}}}}
        --output ${{{{outputs.index}}}}
        --metrics ${{{{outputs.metrics}}}}
        --train-sample {20_000 if pilot else 100_000}
        --batch-size 64
        """,
        inputs={
            "prepared": folder(prepared_uri),
            "model": e5,
        },
        outputs={
            "index": output(e5_index_uri),
            "metrics": output(f"{METRICS}/index-e5"),
        },
    )


# Step 20
if enabled(20):
    for bits in index_bits:
        submit(
            f"20-eval-colbert-{bits}bit",
            f"""
            python ralir_job.py eval-colbert
            --prepared ${{{{inputs.prepared}}}}
            --index ${{{{inputs.index}}}}
            --model ${{{{inputs.model}}}}
            --utility-adapter ${{{{inputs.utility}}}}
            --supervised-adapter ${{{{inputs.supervised}}}}
            --rankings ${{{{outputs.rankings}}}}
            --metrics ${{{{outputs.metrics}}}}
            --max-queries {retrieval_queries}
            --k 100
            --seed 29
            """,
            inputs={
                "prepared": folder(prepared_uri),
                "index": folder(index_uris[bits]),
                "model": colbert,
                "utility": folder(utility_adapter_uri),
                "supervised": folder(supervised_adapter_uri),
            },
            outputs={
                "rankings": output(colbert_ranking_uris[bits]),
                "metrics": output(
                    f"{METRICS}/retrieval-colbert-{bits}bit"
                    f"{bits}bit-{STEP20_ATTEMPT}"
                ),
            },
        )


# Step 21
if enabled(21):
    submit(
        "21-eval-e5",
        f"""
        python ralir_job.py eval-e5
        --prepared ${{{{inputs.prepared}}}}
        --index ${{{{inputs.index}}}}
        --model ${{{{inputs.model}}}}
        --rankings ${{{{outputs.rankings}}}}
        --metrics ${{{{outputs.metrics}}}}
        --max-queries {retrieval_queries}
        --k 100
        --batch-size 32
        --seed 29
        """,
        inputs={
            "prepared": folder(prepared_uri),
            "index": folder(e5_index_uri),
            "model": e5,
        },
        outputs={
            "rankings": output(e5_rankings_uri),
            "metrics": output(f"{METRICS}/retrieval-e5"),
        },
    )


# Step 22: main 2-bit setting
if enabled(22):
    submit(
        "22-rag-evaluation",
        f"""
        python ralir_job.py rag
        --prepared ${{{{inputs.prepared}}}}
        --colbert-rankings ${{{{inputs.colbert_rankings}}}}
        --e5-rankings ${{{{inputs.e5_rankings}}}}
        --model ${{{{inputs.model}}}}
        --predictions ${{{{outputs.predictions}}}}
        --metrics ${{{{outputs.metrics}}}}
        --max-queries {rag_queries}
        --top-contexts 3
        --batch-size 2
        --max-new-tokens 32
        --seed 41
        """,
        inputs={
            "prepared": folder(prepared_uri),
            "colbert_rankings": folder(colbert_ranking_uris[2]),
            "e5_rankings": folder(e5_rankings_uri),
            "model": phi,
        },
        outputs={
            "predictions": output(rag_predictions_uri),
            "metrics": output(f"{METRICS}/rag"),
        },
    )


# Step 23
if enabled(23):
    submit(
        "23-analysis",
        """
        python ralir_job.py analyze
        --metrics ${{inputs.metrics}}
        --predictions ${{inputs.predictions}}
        --output ${{outputs.results}}
        --bootstrap-samples 10000
        --seed 53
        """,
        inputs={
            "metrics": folder(METRICS),
            "predictions": folder(rag_predictions_uri),
        },
        outputs={
            "results": output(f"{ROOT}/results"),
        },
    )

print("\nConfigured run:", RUN_ID)
print("Output root:", ROOT)

Project directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-control/code/Users/estherxin0011/ralir
Worker code: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-control/code/Users/estherxin0011/ralir/job_code/ralir_job.py
Workspace: Reinforcementinfra
DATA: ralir-dpr-raw 1 uri_folder
DATA: ralir-provenance 1 uri_folder
MODEL: ralir-colbertv2 1 custom_model
MODEL: ralir-reward-qwen 1 custom_model
MODEL: ralir-eval-phi 1 custom_model
MODEL: ralir-e5 1 custom_model
Submitted 20-eval-colbert-2bit: nifty_leaf_4bx4d1c6xg
Studio URL: https://ml.azure.com/runs/nifty_leaf_4bx4d1c6xg?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
RunId: nifty_leaf_4bx4d1c6xg
Web View: https://ml.azure.com/runs/nifty_leaf_4bx4d1c6xg?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute